# 8 - Streaming toward global

The eager-load version hit a wall: daily resolution + a large region does not fit in RAM. This version
**never holds the whole cube** - it caches the region once to a disk-backed memmap, then STREAMS small
`40x56` chunks through the model. RAM stays bounded no matter how large the region (or eventually the
globe) gets.

Pipeline:
1. **Cache** the region to a local `(T,H,W)` float32 **memmap** on disk, filled in small time-blocks (RAM
   only ever holds one block). Reused across kernel restarts, so the slow remote read happens once.
2. **Global stats** from a coarse strided read, so every streamed chunk standardizes consistently
   (`build_pace_channels(..., stats=STATS)`).
3. **Stream** ocean-aware `40x56` chunks from the memmap -> `build_pace_channels` -> `tf.data.from_generator`
   -> train. One chunk in memory at a time.
4. **Predict** the domain by streaming a few frames at a time and stitching (`mtg.tiled_predict`), scored vs
   standard persistence.

Run on us-west-2, fresh kernel, updated `mindthegap` (needs the new `stats=` arg on `build_pace_channels`).

## Setup

In [ ]:
%pip install --force-reinstall --no-cache-dir "git+https://github.com/SAFS-Varanasi-Internship/mindthegap.git@troy-branch"
!pip install -qU icechunk

In [ ]:
import os
os.environ.pop("TF_CUDNN_DETERMINISTIC", None)
os.environ.pop("TF_CUDNN_USE_AUTOTUNE", None)
import earthaccess
import icechunk as ic
import numpy as np, pandas as pd
import xarray as xr
import tensorflow as tf
from scipy import ndimage
import matplotlib.pyplot as plt
import cartopy.crs as ccrs, cartopy.feature as cfeature
import mindthegap as mtg

for _g in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(_g, True)
print("mtg has stats arg:", "stats" in mtg.build_pace_channels.__code__.co_varnames)

def create_ds(product="PACE_OCI_L3M_CHL", group="daily/0p1deg/chunks_512"):
    """Open a PACE Icechunk store group as xarray (Eli's helper). Needs AWS us-west-2 + earthaccess login."""
    url = f"https://data.source.coop/fish-pace/pace-oci/inregion/{product}"
    storage = ic.http_storage(url)
    auth = earthaccess.login()
    creds = auth.get_s3_credentials(daac="OBDAAC")
    vc = ic.credentials.containers_credentials({
        "s3://ob-cumulus-prod-public/": ic.credentials.s3_credentials(
            access_key_id=creds["accessKeyId"], secret_access_key=creds["secretAccessKey"],
            session_token=creds["sessionToken"])})
    store = ic.Repository.open(storage, authorize_virtual_chunk_access=vc).readonly_session("main").store
    return xr.open_zarr(store, consolidated=False, group=group, chunks={})

ds = create_ds()
CHL = "chlor_a"
print("global grid:", dict(ds.sizes))

## Region + memmap cache

Now the region can be **large** (we never load it whole). We define it lazily, then fill a disk memmap in
time-blocks. The cache is keyed by name and reused if present - so re-running (or a kernel restart) skips
the slow remote read.

In [ ]:
# --- region can be big now: we stream it, never load it whole ---
LAT_HI, LAT_LO = 30, -30       # a real chunk of the Indian Ocean basin
LON_LO, LON_HI = 40, 110
COMPOSITE_DAYS = 1             # daily = the hard persistence bar; streaming removes the RAM reason to composite

reg = ds[CHL].sel(lat=slice(LAT_HI, LAT_LO), lon=slice(LON_LO, LON_HI))     # lazy, all time
if COMPOSITE_DAYS > 1:
    reg = reg.resample(time=f"{COMPOSITE_DAYS}D").mean()
nlat, nlon = reg.sizes["lat"], reg.sizes["lon"]
reg = reg.isel(lat=slice(0, nlat - nlat % 8), lon=slice(0, nlon - nlon % 8))
times = reg.time.values
T, H, W = reg.sizes["time"], reg.sizes["lat"], reg.sizes["lon"]
print(f"region {H}x{W}, {T} frames  ->  cache is {T*H*W*4/1e9:.2f} GB on DISK (not RAM)")

In [ ]:
CACHE = f"pace_cache/io_{H}x{W}_{T}_c{COMPOSITE_DAYS}.npy"
os.makedirs("pace_cache", exist_ok=True)

if os.path.exists(CACHE):
    mm = np.lib.format.open_memmap(CACHE, mode="r")            # reuse: skip the slow remote read
    print("reusing cache", CACHE, mm.shape)
else:
    mm = np.lib.format.open_memmap(CACHE, mode="w+", dtype="float32", shape=(T, H, W))
    BLK = 20                                                  # frames per remote read; RAM ~ BLK x H x W only
    for t0 in range(0, T, BLK):
        blk = reg.isel(time=slice(t0, t0 + BLK)).values       # (b,H,W) read from the remote store
        mm[t0:t0 + BLK] = np.log(np.where(blk > 0, blk, np.nan)).astype("float32")
        print("cached", min(t0 + BLK, T), "/", T, flush=True)
    mm.flush()
    print("cache built:", CACHE)

# ocean mask WITHOUT loading the cube: OR finite over time in blocks
ocean = np.zeros((H, W), bool)
for t0 in range(0, T, 50):
    ocean |= np.isfinite(mm[t0:t0 + 50]).any(0)
print("ocean:", f"{ocean.mean():.0%}")

In [ ]:
ext = [float(reg.lon.min()), float(reg.lon.max()), float(reg.lat.min()), float(reg.lat.max())]
fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={"projection": ccrs.PlateCarree()})
ax.imshow(np.where(ocean, 1.0, np.nan), extent=ext, origin="upper", transform=ccrs.PlateCarree(), cmap="Blues", vmin=0, vmax=1.5)
ax.add_feature(cfeature.COASTLINE, linewidth=0.5); ax.add_feature(cfeature.BORDERS, linewidth=0.2)
ax.set_title(f"streamed region  {H}x{W}  ({ocean.mean():.0%} ocean, {T} frames)"); plt.show()

## Global standardization stats

Every streamed chunk must standardize with ONE consistent set of stats (else each chunk normalizes to its
own mean and predictions cannot be inverted). We compute them once from a coarse strided read - small
regardless of region size - and pass `stats=STATS` to every per-chunk `build_pace_channels` call.

In [ ]:
tr, va, te = mtg.contiguous_split(T, frac_train=0.6, frac_val=0.2, buf=2)
print("split frames -> train", int(tr.sum()), "val", int(va.sum()), "test", int(te.sum()))

CS = 4                                                        # coarse spatial stride for the stats pass
coarse = np.asarray(mm[:, ::CS, ::CS])                        # (T, H/CS, W/CS) - small, into RAM
_, _, STATS, ORDER = mtg.build_pace_channels(
    coarse, times, tr, n_days=1, cloud_mode="synthetic", coverage=0.2, time_sigma=1.5, seed=0)  # stats=None -> compute
NC = len(ORDER)
Y_MEAN, Y_STD = STATS["CHL"]
del coarse
print("global stats over", NC, "channels; CHL mean/std =", (round(Y_MEAN, 3), round(Y_STD, 3)))

## Streaming generator

An infinite generator: pick an ocean-aware `40x56` chunk, read just that spatial column across all time from
the **memmap** (~6 MB), build its channels with the shared `STATS`, and yield each training frame. One chunk
in memory at a time. A small fixed validation set is built once (eagerly) for the `val_loss` callback.

In [ ]:
CHUNK_H, CHUNK_W = 40, 56
COVERAGE = 0.2
BATCH = 16
STEPS_PER_EPOCH = 200
train_frames = np.where(tr)[0]
val_frames = np.where(va)[0]

def _ocean_chunk(rng):
    for _ in range(100):
        yy = int(rng.integers(0, H - CHUNK_H + 1)); xx = int(rng.integers(0, W - CHUNK_W + 1))
        if ocean[yy:yy + CHUNK_H, xx:xx + CHUNK_W].mean() >= 0.5:
            return yy, xx
    return yy, xx

def _chunk_channels(yy, xx, seed):
    chl_chunk = np.asarray(mm[:, yy:yy + CHUNK_H, xx:xx + CHUNK_W])   # (T,th,tw) from DISK, small
    ch, y, _, _ = mtg.build_pace_channels(chl_chunk, times, tr, n_days=1, cloud_mode="synthetic",
                                          coverage=COVERAGE, time_sigma=1.5, seed=seed, stats=STATS)
    X = np.stack([ch[k] for k in ORDER], -1).astype("float32")       # (T,th,tw,NC)
    Y = mtg.target_with_mask(y)                                      # (T,th,tw,2)
    return X, Y

def gen():
    rng = np.random.default_rng(0)
    while True:
        yy, xx = _ocean_chunk(rng)
        X, Y = _chunk_channels(yy, xx, int(rng.integers(10**9)))     # fresh fake clouds per chunk (augment)
        order = train_frames.copy(); rng.shuffle(order)
        for d in order:
            yield X[d], Y[d]

sig = (tf.TensorSpec((CHUNK_H, CHUNK_W, NC), tf.float32), tf.TensorSpec((CHUNK_H, CHUNK_W, 2), tf.float32))
ds_tr = (tf.data.Dataset.from_generator(gen, output_signature=sig)
         .shuffle(2048)                                       # mix across chunks (gen yields one chunk at a time)
         .batch(BATCH, drop_remainder=True).prefetch(2))

# small fixed validation set (a few chunks x their val frames), built once
vr = np.random.default_rng(123); Xvl, Yvl = [], []
for _ in range(6):
    yy, xx = _ocean_chunk(vr)
    X, Y = _chunk_channels(yy, xx, 7)
    Xvl.append(X[val_frames]); Yvl.append(Y[val_frames])
Xva = np.concatenate(Xvl).astype("float32"); Yva = np.concatenate(Yvl).astype("float32")
del Xvl, Yvl
print("val set", Xva.shape)

## Train

In [ ]:
model = mtg.UNet((None, None, NC))                 # fully-conv
model.compile("adam", loss=mtg.masked_mse, jit_compile=False)   # masked loss, no metric, NO XLA (cuDNN)
es = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True)
model.fit(ds_tr, steps_per_epoch=STEPS_PER_EPOCH, validation_data=(Xva, Yva),
          epochs=40, callbacks=[es], verbose=2)

os.makedirs("models/pace", exist_ok=True)
model.save("models/pace/io_streamed.keras")
print("saved -> models/pace/io_streamed.keras")

## Streamed prediction + score

Predict a few test frames by reading just a 3-frame window (for prev/next) of the WHOLE domain at a time,
building channels with the shared stats, and stitching with `mtg.tiled_predict`. Scored vs standard
persistence (yesterday's real observed pixel). Never holds more than a few frames.

In [ ]:
PRED_TH = max(8, (H // 3 // 8) * 8)                     # prediction chunk ~ third of the domain
PRED_TW = max(8, (W // 3 // 8) * 8)
SCORE = np.where(te)[0]
SCORE = SCORE[SCORE >= 1][:16]                          # a slice of test frames (each needs day-1)
print(f"prediction chunk {PRED_TH}x{PRED_TW} over {H}x{W}, scoring {len(SCORE)} frames")

unet_mae, persist_mae, preds = [], [], {}
for d in SCORE:
    win = np.asarray(mm[d - 1:d + 2])                   # (3,H,W) whole-domain, 3 frames only
    ch, y, _, _ = mtg.build_pace_channels(win, times[d - 1:d + 2], np.ones(win.shape[0], bool),
                                          n_days=1, cloud_mode="synthetic", coverage=COVERAGE,
                                          time_sigma=1.5, seed=1000 + int(d), stats=STATS, land=~ocean)
    Xd = np.stack([ch[k] for k in ORDER], -1)[1]        # middle frame channels (H,W,NC)
    faked = ch["fake_cloud_flag"][1].astype(bool)
    truth, prev = win[1], win[0]                        # log Chl-a day d and d-1
    pred = mtg.tiled_predict(model, Xd, (PRED_TH, PRED_TW), (PRED_TH // 2, PRED_TW // 2)) * Y_STD + Y_MEAN
    preds[int(d)] = pred
    unet_mae.append(mtg.fake_cloud_mae(pred, truth, faked))
    persist_mae.append(mtg.fake_cloud_mae(prev, truth, faked))

u, p = float(np.mean(unet_mae)), float(np.mean(persist_mae))
print(f"\nfake-cloud MAE (log Chl-a) over {len(SCORE)} test frames:")
print(f"   U-Net (streamed + stitched): {u:.4f}")
print(f"   persistence (yesterday):     {p:.4f}")
print("   -> U-Net beats persistence" if u < p else "   -> persistence still ahead")

In [ ]:
# filled map on one test frame
d = int(SCORE[len(SCORE) // 2])
win = np.asarray(mm[d - 1:d + 2])
ch, y, _, _ = mtg.build_pace_channels(win, times[d - 1:d + 2], np.ones(win.shape[0], bool),
                                      n_days=1, cloud_mode="synthetic", coverage=COVERAGE,
                                      time_sigma=1.5, seed=1000 + d, stats=STATS, land=~ocean)
faked = ch["fake_cloud_flag"][1].astype(bool); truth = win[1]
holes = faked | ~np.isfinite(truth)
panels = [("input (fake clouds hidden)", np.where(ocean, np.where(holes, np.nan, truth), np.nan), "viridis"),
          ("U-Net filled (streamed)",    np.where(ocean, preds[d], np.nan),                       "viridis"),
          ("truth (log Chl-a)",          np.where(ocean, truth, np.nan),                          "viridis"),
          ("abs error @ fake",           np.where(faked, np.abs(preds[d] - truth), np.nan),       "magma")]
fig, ax = plt.subplots(1, 4, figsize=(20, 5))
for a, (t, arr, cm) in zip(ax, panels):
    im = a.imshow(arr, cmap=cm, extent=ext, origin="upper"); a.set_title(t, size=10); a.axis("off")
    fig.colorbar(im, ax=a, shrink=0.7)
plt.suptitle(f"Streamed gap-fill over {H}x{W}, test frame {d}", size=12); plt.tight_layout(); plt.show()

## Next steps

- **Scale the region toward the full basin / global** by just widening `LAT/LON` - the memmap grows on
  disk, RAM stays flat. At large latitudes, swap the chunk sampler for `cos(lat)`-weighted (globe) sampling.
- **Throughput:** the generator rebuilds channels per chunk; if training is I/O- or CPU-bound, cache a pool
  of pre-built chunks or widen `prefetch`.
- **Positional channels** (lat/lon) once the domain spans many chlorophyll regimes.